# Unassigned-cell curvature and neighborhood context

**Role:** Analysis.

Examine naturally unassigned cells as an observed identity: their prevalence, measured curvature, neighboring fates and model prediction behavior. Restore the recorded exclusive-fate cohort and checkpoints, with the study sequence defined below. Contrast naturally unassigned cells with zeroed source cells to assess the reference-state dependence of marker ablation. The analysis reports support across size and does not interpret zeroing as biological cell removal.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.


## Observed figure definitions

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.analysis.spatial.neighborhoods import BIN_LABELS, MARKERS

def observed_axis(ax,ylabel):
    ax.set_xticks(range(6),BIN_LABELS,rotation=30)
    ax.set_xlabel('Observed cell count N (size bins)')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=.15)

def observed_curve(ax,table,metric,label,color,**kw):
    t=table[(table.metric==metric)&table.supported].sort_values('size_bin')
    # Reindex so unsupported intermediate bins are gaps rather than connected lines.
    t=t.set_index('size_bin').reindex(range(6))
    ax.plot(t.index,t['mean'],marker='o',color=color,label=label,**kw)
    ax.fill_between(t.index,t.low,t.high,color=color,alpha=.12)

## Unassigned figure definitions

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.analysis.spatial.neighborhoods import MARKERS, BIN_LABELS

UNASSIGNED_COLORS = {
    'Unmarked': '#7b3294',  # Display color for Unmarked.
    'KI67': '#b35806',  # Display color for KI67.
    'LGR5': '#2166ac',  # Display color for LGR5.
    'AldoB': '#359449',  # Display color for AldoB.
    'Agr2': '#d45a31',  # Display color for Agr2.
    'Mixed': '.4',  # Display color for Mixed.
}  # Display colors assigned to groups or model variants.

def unassigned_label(name):return 'Unassigned' if name=='Unmarked' else name

def unassigned_read(out,name):return pd.read_csv(Path(out)/f'{name}.csv')

def unassigned_population(out):
    fig,axs=plt.subplots(1,3,figsize=(12,3.8),layout='constrained')
    observed_curve(axs[0],unassigned_read(out,'prevalence_by_size'),'fraction_unassigned','Unassigned',UNASSIGNED_COLORS['Unmarked'])
    profiles=unassigned_read(out,'profiles_by_size')
    for marker in ['Unmarked','KI67','LGR5','AldoB']:
        part=profiles[profiles.marker==marker]
        observed_curve(axs[1],part,'curvature_rank',unassigned_label(marker),UNASSIGNED_COLORS[marker])
        observed_curve(axs[2],part,'negative',unassigned_label(marker),UNASSIGNED_COLORS[marker])
    for ax,y,t in zip(axs,['Fraction of cells','Mean within-organoid curvature percentile','Fraction with measured K < 0'],
        ['A  Unassigned-cell prevalence','B  Measured curvature rank','C  Negative Gaussian curvature']):observed_axis(ax,y);ax.set_title(t)
    axs[1].legend(fontsize=8);axs[1].axhline(.5,color='.7',ls=':')
    return fig

def unassigned_context(out):
    fig,axs=plt.subplots(1,3,figsize=(12,4.2),layout='constrained')
    qc=unassigned_read(out,'curvature_qc_summary');qc=qc[qc.marker=='Unmarked']
    observed_curve(axs[0],qc,'curvature_norm','Raw graph curvature','.45')
    observed_curve(axs[0],qc,'clean_curvature_norm','Configured outlier interpolation',UNASSIGNED_COLORS['Unmarked'])
    contexts=unassigned_read(out,'unassigned_context_curves')
    for name in ['Unmarked','KI67','LGR5','AldoB','Mixed']:
        observed_curve(axs[1],contexts[contexts.context==name],'curvature_rank',unassigned_label(name),UNASSIGNED_COLORS[name])
    profiles=unassigned_read(out,'profiles_by_size')
    for marker in ['Unmarked','KI67','LGR5']:
        observed_curve(axs[2],profiles[profiles.marker==marker],'fraction_neck_band',unassigned_label(marker),UNASSIGNED_COLORS[marker])
    for a,y,t in zip(axs,['Measured K × area / (4π)','Curvature percentile of unassigned center','Fraction in qualified neck band'],
        ['A  Outlier sensitivity of unassigned means','B  Unassigned cells, grouped by neighbors','C  Only circumference-qualified territories']):observed_axis(a,y);a.set_title(t)
    axs[0].legend(fontsize=7);axs[1].legend(title='Strict majority of hop-1 neighbors',fontsize=7,title_fontsize=7)
    axs[2].legend(fontsize=8)
    return fig

def unassigned_neighborhoods(out):
    t=unassigned_read(out,'profiles_by_size');t=t[t.marker=='Unmarked']
    fig,axs=plt.subplots(2,2,figsize=(11,7.3),layout='constrained')
    for i,hop in enumerate([1,2]):
        for j,kind in enumerate(['fraction','enrichment']):
            a=axs[i,j];matrix=[]
            for marker in MARKERS:
                p=t[(t.metric==f'ring{hop}_{kind}_{marker}')&t.supported].set_index('size_bin')
                matrix.append(p['mean'].reindex(range(6)).to_numpy()*100)
            matrix=np.asarray(matrix);lim=np.nanmax(np.abs(matrix))
            im=a.imshow(matrix,aspect='auto',cmap='Blues' if j==0 else 'RdBu_r',vmin=0 if j==0 else -lim,vmax=lim)
            for row in range(8):
                for col in range(6):
                    v=matrix[row,col]
                    if np.isfinite(v):a.text(col,row,f'{v:.0f}',ha='center',va='center',fontsize=8,color='white' if abs(v)>.65*lim else 'black')
            a.set_yticks(range(8),[unassigned_label(m) for m in MARKERS]);a.set_xticks(range(6),BIN_LABELS,rotation=30)
            a.set_xlabel('Observed N');a.set_title(f'Hop {hop}: '+('neighbor fractions' if j==0 else 'excess over organoid composition'))
            fig.colorbar(im,ax=a,label='Percent' if j==0 else 'Percentage points')
    return fig

def unassigned_neighbor_associations(out):
    raw=unassigned_read(out,'neighbor_summary_unmatched');matched=unassigned_read(out,'neighbor_summary_degree_matched')
    fig,axs=plt.subplots(1,3,figsize=(12,4.8),layout='constrained')
    matrices=[]
    for table,hop in [(raw,1),(raw,2),(matched,1)]:
        p=table[(table.hop==hop)&(table.metric=='curvature_rank')&table.supported]
        matrices.append(p.pivot(index='marker',columns='size_bin',values='mean').reindex(index=MARKERS,columns=range(6)).to_numpy()*100)
    limit=max(np.nanmax(np.abs(m)) for m in matrices)
    for a,m,t in zip(axs,matrices,['A  Hop 1: with − without unassigned','B  Hop 2: with − without unassigned','C  Hop 1: also match exact degree']):
        im=a.imshow(np.ma.masked_invalid(m),aspect='auto',cmap='RdBu_r',vmin=-limit,vmax=limit)
        for i in range(8):
            for j in range(6):
                if np.isfinite(m[i,j]):a.text(j,i,f'{m[i,j]:.0f}',ha='center',va='center',fontsize=8,color='white' if abs(m[i,j])>.65*limit else 'black')
        a.set_yticks(range(8),[unassigned_label(v) for v in MARKERS]);a.set_xticks(range(6),BIN_LABELS,rotation=45)
        a.set_xlabel('Observed N');a.set_title(t)
    axs[0].set_ylabel('Unchanged center fate')
    fig.colorbar(im,ax=list(axs),label='Difference in measured curvature percentile (percentage points)',shrink=.8)
    return fig

def unassigned_prediction_checks(out):
    table=unassigned_read(out,'curvature_qc_summary');neighbors=unassigned_read(out,'neighbor_qc_summary')
    fig,axs=plt.subplots(1,3,figsize=(12,4.2),layout='constrained')
    for marker in ['Unmarked','KI67','LGR5','AldoB']:
        p=table[table.marker==marker]
        observed_curve(axs[0],p,'clean_curvature_centered',unassigned_label(marker)+' measured',UNASSIGNED_COLORS[marker])
        observed_curve(axs[0],p,'prediction_centered',unassigned_label(marker)+' predicted',UNASSIGNED_COLORS[marker],ls='--')
    p=table[table.marker=='Unmarked'].set_index(['metric','size_bin'])
    for metric,name,color in [('squared_error','Raw measured targets','.5'),('clean_squared_error','Interpolated targets',UNASSIGNED_COLORS['Unmarked'])]:
        part=p.loc[metric].reindex(range(6));axs[1].plot(part.index,np.sqrt(part['mean']),marker='o',label=name,color=color)
    axs[1].set_yscale('log')
    for marker in ['LGR5','KI67','AldoB']:
        p=neighbors[(neighbors.marker==marker)&(neighbors.hop==1)]
        observed_curve(axs[2],p,'clean_curvature_norm',unassigned_label(marker)+' measured',UNASSIGNED_COLORS[marker])
        observed_curve(axs[2],p,'prediction_centered',unassigned_label(marker)+' predicted',UNASSIGNED_COLORS[marker],ls='--')
    for a,y,t in zip(axs,['Curvature relative to organoid mean × area / (4π)','Within-organoid RMSE (normalized units; log scale)',
        'With − without unassigned neighbor (normalized units)'],['A  Same observed cells: mean prediction','B  Unassigned-cell prediction error','C  Does the model reproduce neighbor associations?']):observed_axis(a,y);a.set_title(t)
    axs[0].legend(fontsize=6,ncol=2);axs[1].legend(fontsize=7);axs[2].legend(fontsize=6,ncol=2)
    return fig

def unassigned_ablation_reference(out):
    table=unassigned_read(out,'ablation_matrix')
    fig,axs=plt.subplots(1,2,figsize=(12,5),layout='constrained')
    matrices=[]
    for hop in [1,2]:
        p=table[(table.hop==hop)&table.supported]
        matrices.append(p.pivot(index='recipient',columns='source_marker_name',values='mean').reindex(index=MARKERS,columns=MARKERS[:-1]).to_numpy())
    limit=max(np.nanmax(np.abs(m)) for m in matrices)
    for ax,m,hop in zip(axs,matrices,[1,2]):
        im=ax.imshow(np.ma.masked_invalid(m),aspect='auto',cmap='RdBu_r',vmin=-limit,vmax=limit)
        ax.set_xticks(range(7),MARKERS[:-1],rotation=45,ha='right');ax.set_yticks(range(8),[unassigned_label(v) for v in MARKERS])
        ax.set_title(f'Hop {hop}: source marker → unassigned vector')
        ax.set_xlabel('Marker cleared from one source cell');ax.set_ylabel('Unchanged recipient fate')
        for i in range(8):
            for j in range(7):
                if np.isfinite(m[i,j]):ax.text(j,i,f'{m[i,j]:+.2f}',ha='center',va='center',fontsize=7,color='white' if abs(m[i,j])>.65*limit else 'black')
    fig.colorbar(im,ax=list(axs),label='Predicted K(after) − K(before), normalized by measured area',shrink=.8)
    return fig

# Unassigned cells: curvature, neighborhoods and the ablation reference state

**An all-zero fate vector is an observed input state, not the absence of a cell.** This notebook studies what that state represents in the dataset and in the trained exclusive-marker FiLM model.

“Unassigned” means negative for all seven measured binary markers. It does not identify a single biological lineage, a dead cell, or a missing segmentation. Cached tables use the label `Unmarked`; figures use **Unassigned**. The encoding-transition audit distinguishes naturally zero vectors from any vectors made zero by exclusivity.

We analyze all 1,100 organoids retained in the existing experiment, 477,789 cells, five folds and three trained seeds. All inference is held out by organoid, at its **observed N**. There is no retraining or counterfactual size sweep in this notebook. It covers (1) the unassigned population, (2) its context and measured curvature, (3) associations with other cell types, (4) held-out model behavior, and (5) what marker-to-zero ablations actually compare.

N bins and collection labels follow the preceding observed-neighborhood study. Size is not interchangeable with collection time. Means give organoids equal weight after averaging cells; bands are 95% organoid bootstrap intervals, conditional on these datasets and fitted models. Fewer than ten contributing organoids gives a blank in the figures.

In [ ]:
from src.artifacts.provenance import workflow_fingerprint
from dataclasses import dataclass, asdict
from pathlib import Path
import hashlib
import json
import pickle
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Batch
from src.data.io import build_pyg_graph
from src.analysis.size_conditioning.cohort_inputs import make_model
from src.analysis.spatial.neighborhoods import MARKERS, ObservedConfig, bootstrap_summary
from src.analysis.spatial.unassigned import UnassignedConfig, majority_context, neighbor_contrasts, _stats, observed_tables, infer_fold, prediction_tables, ablation_tables, source_context_comparison, add_curvature_qc
WORKFLOW_SOURCE = PROJECT_ROOT / 'experiments/neighborhoods/unassigned_cells.ipynb'  # Notebook used when recording the code fingerprint.


### Run

The study sequence is defined here; shared numerical operations are imported above.

In [ ]:
def run(root,config=UnassignedConfig(),device='cpu'):
    root=Path(root)
    run_dir=root/'results_experiments/size_conditioned_exclusive/20260914_ordered_exclusive_film_depth2/exclusive'
    observed=root/'results_experiments/ki67_observed_neighborhoods/exclusive_v2_profile_necks'
    out=root/'results_experiments/unassigned_cells/exclusive_v1';out.mkdir(parents=True,exist_ok=True)
    settings=json.loads((run_dir/'settings.json').read_text());data=root/'training_data'/settings['DATASET_NAME']
    signature=json.loads(json.dumps(dict(config=asdict(config),model_run=str(run_dir),observed_source=str(observed))))
    if (out/'settings.json').exists() and json.loads((out/'settings.json').read_text())!=signature:raise ValueError('Incompatible saved configuration')
    (out/'settings.json').write_text(json.dumps(signature,indent=2))
    if (out/'complete.json').exists():
        add_curvature_qc(root,out,config)
        source_context_comparison(out,config)
        return out
    nodes=pd.read_csv(observed/'nodes.csv.gz');organs=pd.read_csv(observed/'organoids.csv')
    nodes['context']=majority_context(nodes)
    audit=pd.read_csv(run_dir/'tables/encoding_transition_counts.csv')
    zeros=audit[audit.exclusive_marker=='unmarked']
    assert zeros.n_cells.sum()==nodes.marker.eq('Unmarked').sum()
    provenance=zeros.groupby('original_markers').n_cells.sum().reset_index()
    provenance.to_csv(out/'unassigned_provenance.csv',index=False)
    if not (out/'observed_complete.json').exists():
        observed_tables(nodes,organs,out,config)
        (out/'observed_complete.json').write_text('{}')
    for fold in config.folds:infer_fold(root,run_dir,data,settings,fold,nodes,config,out,device)
    prediction_tables(nodes,out,config)
    ablation_tables(nodes,organs,run_dir,out,config)
    checks=[json.loads(p.read_text()) for p in out.glob('fold_*/complete.json')]
    (out/'complete.json').write_text(json.dumps(dict(cells=len(nodes),unassigned_cells=int(nodes.marker.eq('Unmarked').sum()),
        organoids=len(organs),unassigned_organoids=int(nodes[nodes.marker=='Unmarked'].organoid_str.nunique()),
        checkpoints=len(checks),maximum_saved_baseline_error=max(c['max_saved_baseline_error'] for c in checks)),indent=2))
    add_curvature_qc(root,out,config)
    source_context_comparison(out,config)
    return out

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else PROJECT_ROOT  # Repository root used to resolve data and result paths.
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.analysis.spatial.unassigned import UnassignedConfig

# Study-specific functions are defined in this notebook.

CONFIG = UnassignedConfig(
    # Support thresholds
    minimum_cells_per_group=3,  # Minimum cells per within-organoid comparison group.

    # Statistical summaries
    minimum_organoids=10,  # Minimum organoid support for each comparison.
    bootstrap_draws=1000,  # Number of organoid bootstrap resamples.

    # Inference resources
    graph_batch_size=8,  # Number of full graphs per inference batch.
)  # Settings for the observed-cell or intervention study.

# Paths and outputs
OUTPUT = ROOT / 'results_experiments/unassigned_cells/exclusive_v1'  # Directory containing the selected analysis results.

# Execution switches
RUN_ANALYSIS = False  # Recompute analysis tables instead of only reading saved results.
if RUN_ANALYSIS:
    import torch
    from threadpoolctl import threadpool_limits
    torch.set_num_threads(4)
    with threadpool_limits(limits=4):
        OUTPUT = run(ROOT, CONFIG, device='cuda' if torch.cuda.is_available() else 'cpu')
assert (OUTPUT / 'complete.json').exists()
assert (OUTPUT / 'curvature_qc_complete.json').exists()
from dataclasses import asdict
assert json.loads((OUTPUT / 'settings.json').read_text())['config'] == json.loads(json.dumps(asdict(CONFIG)))
FIGURES = OUTPUT / 'figures'  # Destination for the generated figures.
FIGURES.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':9, 'axes.titlesize':10, 'figure.dpi':120})
def show(fig, name):
    fig.savefig(FIGURES / f'{name}.png', dpi=170, bbox_inches='tight')
    fig.savefig(FIGURES / f'{name}.pdf', bbox_inches='tight')
    plt.show(); plt.close(fig)
print(json.dumps(json.loads((OUTPUT / 'complete.json').read_text()), indent=2))
display(pd.read_csv(OUTPUT / 'unassigned_provenance.csv'))


## Figure 1 — How common are unassigned cells, and what curvature do they have?

The audit finds **125,148 unassigned cells**, all already negative for the marker panel before exclusivity. None was newly made unassigned by the exclusivity rules. They account for 26.2% of all cells and occur in 1,063 organoids. This pooled cell fraction differs from the equally weighted organoid fractions plotted in A.

B ranks each cell's measured Gaussian curvature among all cells in the same organoid. C shows the fraction with negative measured curvature. These figures characterize the entire unassigned population, not only unassigned cells neighboring LGR5.

In [ ]:
show(unassigned_population(OUTPUT), 'figure_1_unassigned_population')

## Figure 2 — Heterogeneity, outliers and anatomical context

A checks sensitivity of unassigned mean curvature to the **same configured outlier-interpolation routine used in the training pipeline** (global 0.5th/99.5th-percentile thresholds, neighbor interpolation). Raw measurements are retained and displayed separately. Reapplying this routine is a processing sensitivity check, not proof that every changed value is erroneous. Both curves use K×A/(4π), with measured organoid area. The isolated depression of the raw mean in N=150–249 is particularly sensitive to extreme values; do not interpret it as a biological minimum without checking the rank-based results.

B groups unassigned centers by the strict majority (>50%) of their immediate neighbors. If no fate has a majority, the context is Mixed; an unassigned-majority neighborhood is its own context. **Every center in this panel remains unassigned.** Groups can differ in spatial position and other features, so this is not an effect estimate of the neighboring fate. Rare contexts have insufficient support and remain in the exported tables rather than being forced into a figure.

C uses only the circumference-qualified crypt territories established in the preceding notebook: a supported minimum or flat section near s=1 and at least ten cells assigned inside s≤1. Original nearest-crypt assignments are preserved. Nonqualifying and undetected crypts are excluded from position estimates. The neck band is 0.8–1.2; low-N support is insufficient. These anatomical estimates are conditional on detectable, qualified crypts.

In [ ]:
show(unassigned_context(OUTPUT), 'figure_2_context_and_quality')
display(pd.read_csv(OUTPUT / 'context_support.csv').rename(columns={'context':'majority_neighbor_context'}))
print('Outlier-processing audit:', json.loads((OUTPUT / 'curvature_qc_complete.json').read_text()))

## Figure 3 — What surrounds an unassigned cell?

Exact hop-1 and hop-2 rings exclude the center; hop 2 also excludes hop 1. Left panels show neighbor marker fractions. Right panels subtract the expected composition of other cells in the same organoid, excluding the center, in percentage points. This controls descriptively for marker abundance; it is not a fitted spatial null model or a test of cell–cell attraction.

Concentration of unassigned cells next to one another matters for interpreting ablations: naturally unassigned cells need not occupy the same contexts as the marker-positive cells we convert to zero.

In [ ]:
show(unassigned_neighborhoods(OUTPUT), 'figure_3_unassigned_neighborhoods')

## Figure 4 — How does nearby unassigned tissue correlate with other cell types?

For each center fate, compare cells with at least one unassigned cell in the selected ring against cells without one, **within the same organoid and center fate**. Both groups must contain at least three cells. The heatmaps show the mean difference in measured curvature percentile; negative means the centers near unassigned cells have lower curvature. Centers are counted once per comparison, not once for every incident edge.

C additionally compares only cells of the same exact hop-1 degree. Within each organoid, eligible degree-stratum contrasts are weighted by the smaller of the two group counts, then organoids are weighted equally. This checks whether simple degree differences explain the association. It does not match every aspect of neighborhood composition or position. At hop 2, few cells lack an unassigned neighbor, so blank cells can reflect lack of a comparison group rather than absence of an association.

Pointwise intervals and support counts are in `neighbor_summary_unmatched.csv` and `neighbor_summary_degree_matched.csv`. These are exploratory associations, not causal effects. Replacing a neighbor's fate necessarily changes composition; an “unassigned effect” cannot be isolated while holding every other fate count and degree fixed.

In [ ]:
show(unassigned_neighbor_associations(OUTPUT), 'figure_4_other_cell_types')

## Figure 5 — Does the trained model reproduce these patterns?

Each cell is predicted by the three checkpoints whose training fold excludes its organoid. Seeds are averaged per cell before summarizing organoids. The GNN input remains the exclusive fate panel and log N, with no geometry passed to its head.

A compares measured and predicted curvature **relative to each organoid's own mean**, normalized by measured area. Solid lines use the interpolated targets; dashed lines use held-out predictions. Centering removes the constant baseline, allowing comparison without attributing the global baseline to the GNN. B shows unassigned-cell RMSE for this centered prediction, separately against raw and interpolated targets. It is not total-curvature validation MSE, and the interpolation thresholds reproduce the original cohort-wide preprocessing convention rather than introducing a new train-only evaluation protocol.

C compares the measured and predicted within-organoid differences between marker-positive cells with and without unassigned hop-1 neighbors. The measured curve uses interpolated targets here; raw and rank-based comparisons remain available. Similar mean predictions do not establish accurate cell-level predictions or validate an ablation as a biological intervention.

In [ ]:
show(unassigned_prediction_checks(OUTPUT), 'figure_5_heldout_model')

## Figure 6 — Every exclusive single-marker removal is a marker-to-unassigned edit

These are the **existing saved ablation results**, evaluated at observed N, not new deletions or fate assignments. The source's sole positive feature is set to zero while its node, edges, neighbors, N and recipient identity stay unchanged. Each source was verified to have the indicated exclusive fate. The y-axis identifies the recipient whose curvature prediction is read out; recipients can themselves be unassigned.

Colors show predicted K(after)−K(before), normalized by the actual organoid area. We average cases and seeds within organoids before weighting organoids equally. Pair eligibility and sample support differ, so the matrix is not a matched marker-importance ranking. It pools observed sizes; `ablation_by_size.csv` retains the size dependence. In particular, this pooled observed-size KI67 value must not be substituted for the earlier fixed-neighborhood size sweep.

A positive value says **replacing this marker vector by zero in these contexts raises the prediction**. It does not measure an intrinsic curvature contribution of that marker, an effect of removing a cell, or a general property of all naturally unassigned cells.

In [ ]:
show(unassigned_ablation_reference(OUTPUT), 'figure_6_marker_to_unassigned')

## Is the replacement state encountered in the same contexts?

The following table compares the fraction of unassigned hop-1 neighbors around ablated source cells with that around naturally unassigned cells **in the same organoid**. Ablated source cells are deduplicated across recipients, hops and seeds before comparison. Only organoids containing both groups contribute. Positive/negative differences indicate different neighborhood composition; this is a limited context check, not a formal out-of-distribution test.

The source sample comes from the existing ablation experiment, so it is not a census of all marker-positive cells. A zero vector can be common in the training data while a particular zero-vector neighborhood remains unusual.

In [ ]:
support = pd.read_csv(OUTPUT / 'ablation_source_context.csv')
display(support[support.metric == 'difference'][['source_marker_name','mean','low','high','n_organoids']].rename(
    columns={'mean':'source_minus_natural_unassigned_neighbor_fraction'}))


## Why an all-zero input node is not computationally silent

A naturally unassigned node and a marker-ablated node have identical **initial** feature vectors. Their final embeddings need not match because their neighbors, edges and organoid size differ.

In this two-layer GIN, an unassigned node's first update aggregates marker signals from its neighbors and passes through learned linear maps, normalization, FiLM and nonlinearities. Its hidden state can therefore be nonzero even though its own input is zero. In the second layer, that state is passed to neighboring cells. The node also remains part of graph connectivity and the supplied organoid count. Setting an already-unassigned input to zero is a no-op; deleting the node or blocking its messages would be a different intervention.

The saved `prediction_curves.csv` includes the mean final local-embedding norm for unassigned cells as a numerical check. A nonzero norm establishes that zero input is not zero hidden state; it is not itself a measure of biological or predictive importance.

The present study therefore describes **associations around naturally unassigned cells and model responses to marker-to-zero edits**. It cannot establish which unmeasured lineages or marker-detection failures produced the unassigned population, nor whether those cells mechanically cause neighboring curvature.

In [ ]:
report = OUTPUT / 'REPORT.md'
if report.exists(): display(Markdown(report.read_text()))